In [2]:
import glob
import shutil
import time

import win32com.client  # Windows COM 인터페이스 사용
import xml.etree.ElementTree as ET
import pandas as pd

import os.path

# =========================================================
# .mer → parquet 변환 함수
# =========================================================
def convert_mer_to_parquet(mer_file, delete_original=True):
    try:
        print(f"[변환 시작] {mer_file}")

        with open(mer_file, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()

        data_start_idx = None
        for i, line in enumerate(lines):
            if "Measurem." in line:
                data_start_idx = i
                break

        if data_start_idx is None:
            print(f"[건너뜀] 데이터 헤더를 찾지 못함: {mer_file}")
            return

        df = pd.read_csv(
            mer_file,
            sep=";",
            skiprows=data_start_idx,
            encoding="utf-8",
            engine="python"
        )

        df.columns = [str(col).strip() for col in df.columns]

        parquet_path = mer_file.replace(".mer", ".parquet")

        df.to_parquet(
            parquet_path,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"[저장 완료] {parquet_path}")
        """
        if delete_original:
            os.remove(mer_file)
            print(f"[원본 삭제 완료] {mer_file}")
        """
    except Exception as e:
        print(f"[변환 실패] {mer_file}")
        print(e)

# 입력 파라미터
###################################################################################################################
network_path = r"C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\진입구간\화성~서울(140-진입구간)_260518_001.mer"

convert_mer_to_parquet(network_path, delete_original=True)



[변환 시작] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\진입구간\화성~서울(140-진입구간)_260518_001.mer
[저장 완료] C:\Vissim_Workspace\시나리오(140-유고지점 Uniform Random)_260518\진입구간\화성~서울(140-진입구간)_260518_001.parquet
